# Session 4: Common Ports, Protocols and Services

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-04-ports-protocols-services.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-04_ports.py`

In [1]:
"""Session 4 - Well-known ports: lookup, range classification and a
risk note for each service (what a defender should check).

Run:  python session-04_ports.py
"""

SERVICES = {
    20: ("FTP data", "TCP", "cleartext; prefer SFTP/FTPS"),
    21: ("FTP control", "TCP", "cleartext logins; prefer SFTP"),
    22: ("SSH / SFTP / SCP", "TCP", "use keys, disable root login"),
    23: ("Telnet", "TCP", "cleartext: replace with SSH"),
    25: ("SMTP", "TCP", "block outbound from PCs to stop spam bots"),
    53: ("DNS", "TCP/UDP", "UDP queries, TCP zone transfers: restrict AXFR"),
    67: ("DHCP server", "UDP", "enable DHCP snooping on switches"),
    68: ("DHCP client", "UDP", ""),
    69: ("TFTP", "UDP", "no authentication; lab/config use only"),
    80: ("HTTP", "TCP", "redirect to HTTPS"),
    110: ("POP3", "TCP", "use POP3S 995"),
    123: ("NTP", "UDP", "accurate time is essential for logs"),
    143: ("IMAP", "TCP", "use IMAPS 993"),
    161: ("SNMP", "UDP", "use SNMPv3; change 'public' community"),
    162: ("SNMP trap", "UDP", ""),
    389: ("LDAP", "TCP/UDP", "use LDAPS 636"),
    443: ("HTTPS", "TCP", "keep TLS 1.2+ only"),
    445: ("SMB", "TCP", "never expose to the internet"),
    514: ("Syslog", "UDP", "consider TLS syslog 6514"),
    636: ("LDAPS", "TCP", ""),
    993: ("IMAPS", "TCP", ""),
    995: ("POP3S", "TCP", ""),
    1433: ("MS SQL Server", "TCP", "never expose to the internet"),
    3306: ("MySQL", "TCP", "bind to localhost/app subnet"),
    3389: ("RDP", "TCP", "put behind VPN + MFA"),
    5060: ("SIP", "TCP/UDP", "VoIP signalling"),
}


def port_range(p: int) -> str:
    if 0 <= p <= 1023: return "well-known"
    if p <= 49151: return "registered"
    if p <= 65535: return "dynamic/ephemeral"
    return "invalid"


def audit(open_ports: list[int]) -> None:
    """Review a list of open ports found on one of YOUR lab hosts."""
    risky = {21, 23, 69, 445, 1433, 3306, 3389, 161}
    for p in sorted(open_ports):
        name, proto, note = SERVICES.get(p, ("unknown", "?", "identify the process using it"))
        flag = "REVIEW" if p in risky else "ok"
        print(f"  {p:>5}/{proto:<7} {name:<16} {port_range(p):<17} {flag:<6} {note}")


if __name__ == "__main__":
    print("Port ranges:")
    for p in (22, 443, 8080, 3389, 50000, 70000):
        print(f"  {p:>5} -> {port_range(p)}")
    print("\nAudit of lab server 'web01' (ports reported by the admin's own inventory):")
    audit([22, 80, 443, 3306, 23, 8080])
    secure_pairs = [(80, 443), (21, 22), (23, 22), (110, 995), (143, 993), (389, 636)]
    print("\nInsecure -> secure replacements:")
    for old, new in secure_pairs:
        print(f"  {SERVICES[old][0]:<12} {old:>4}  ->  {SERVICES[new][0]:<16} {new}")

Port ranges:
     22 -> well-known
    443 -> well-known
   8080 -> registered
   3389 -> registered
  50000 -> dynamic/ephemeral
  70000 -> invalid

Audit of lab server 'web01' (ports reported by the admin's own inventory):
     22/TCP     SSH / SFTP / SCP well-known        ok     use keys, disable root login
     23/TCP     Telnet           well-known        REVIEW cleartext: replace with SSH
     80/TCP     HTTP             well-known        ok     redirect to HTTPS
    443/TCP     HTTPS            well-known        ok     keep TLS 1.2+ only
   3306/TCP     MySQL            registered        REVIEW bind to localhost/app subnet
   8080/?       unknown          registered        ok     identify the process using it

Insecure -> secure replacements:
  HTTP           80  ->  HTTPS            443
  FTP control    21  ->  SSH / SFTP / SCP 22
  Telnet         23  ->  SSH / SFTP / SCP 22
  POP3          110  ->  POP3S            995
  IMAP          143  ->  IMAPS            993
  LDAP      

## `session-04_local_service.py`

In [2]:
"""Session 4 - Sockets on localhost: how a port, a protocol and a service fit
together. Starts a tiny TCP echo service and a UDP time service on 127.0.0.1,
talks to them as a client, then checks which of a few local ports are open.

Lab-safe: everything stays on the loopback interface of your own machine.
Run:  python session-04_local_service.py
"""
import socket
import threading
import time

HOST = "127.0.0.1"


def tcp_echo_server(sock: socket.socket) -> None:
    conn, addr = sock.accept()                  # 3-way handshake already done
    with conn:
        data = conn.recv(1024)
        conn.sendall(b"ECHO: " + data)


def udp_time_server(sock: socket.socket) -> None:
    data, addr = sock.recvfrom(1024)            # no connection, just a datagram
    sock.sendto(b"TIME 12:00:00", addr)


def is_open(port: int) -> bool:
    """TCP connect check, the same thing a firewall test does. Local only."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.3)
        return s.connect_ex((HOST, port)) == 0


if __name__ == "__main__":
    # --- TCP: connection-oriented, reliable, ordered ---
    tcp = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    tcp.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    tcp.bind((HOST, 0))                          # 0 = let the OS pick a free port
    tcp.listen(1)
    tcp_port = tcp.getsockname()[1]
    threading.Thread(target=tcp_echo_server, args=(tcp,), daemon=True).start()

    with socket.create_connection((HOST, tcp_port)) as c:
        local = c.getsockname()[1]
        c.sendall(b"hello over TCP")
        reply = c.recv(1024).decode()
    print(f"TCP  client port {local} is {'ephemeral' if local > 1023 else 'low'}; "
          f"server port is in the dynamic range")
    print(f"TCP  reply: {reply}")

    # --- UDP: connectionless, best-effort ---
    udp = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    udp.bind((HOST, 0))
    udp_port = udp.getsockname()[1]
    threading.Thread(target=udp_time_server, args=(udp,), daemon=True).start()
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as u:
        u.settimeout(1)
        u.sendto(b"what time?", (HOST, udp_port))
        print(f"UDP  reply: {u.recvfrom(1024)[0].decode()}")

    # --- Which local ports are listening? Start one more listener to compare ---
    extra = socket.socket(); extra.bind((HOST, 0)); extra.listen(1)
    listening = extra.getsockname()[1]
    time.sleep(0.1)
    closed = listening + 1 if listening < 65535 else listening - 1
    for p in (listening, closed):
        print(f"Port {'<listening port>' if p == listening else '<next port>'}: "
              f"{'OPEN' if is_open(p) else 'closed (connection refused)'}")
    extra.close(); tcp.close(); udp.close()

TCP  client port 40202 is ephemeral; server port is in the dynamic range
TCP  reply: ECHO: hello over TCP
UDP  reply: TIME 12:00:00
Port <listening port>: OPEN
Port <next port>: closed (connection refused)


## `session-04_ntp_offset.py`

In [3]:
"""Session 4 - How NTP works out the clock offset and the round-trip delay
from four timestamps, and why log timestamps go wrong when clocks drift.

t1 = client sends request    (client clock)
t2 = server receives it      (server clock)
t3 = server sends reply      (server clock)
t4 = client receives reply   (client clock)
offset = ((t2 - t1) + (t3 - t4)) / 2
delay  = (t4 - t1) - (t3 - t2)
Run:  python session-04_ntp_offset.py
"""


def ntp(t1, t2, t3, t4):
    offset = ((t2 - t1) + (t3 - t4)) / 2
    delay = (t4 - t1) - (t3 - t2)
    return offset, delay


if __name__ == "__main__":
    # Times in milliseconds. The client clock is really 250 ms BEHIND the server.
    samples = [
        ("LAN server",        1000.0, 1255.0, 1256.0, 1012.0),
        ("Internet server",   2000.0, 2290.0, 2291.0, 2081.0),
        ("Asymmetric path",   3000.0, 3310.0, 3311.0, 3071.0),
    ]
    print(f"{'Source':<18}{'offset ms':>10}{'delay ms':>10}")
    for name, t1, t2, t3, t4 in samples:
        o, d = ntp(t1, t2, t3, t4)
        print(f"{name:<18}{o:>10.1f}{d:>10.1f}")
    print("\nStratum: 0 = atomic/GPS clock, 1 = server wired to it, 2 = syncs from stratum 1 ...")

    # Why it matters: two devices log the same attack, one clock is 4 minutes slow.
    print("\nCorrelating firewall and server logs (server clock 240 s slow):")
    fw = [("09:15:02", "firewall: allowed 198.51.100.7 -> web01:443")]
    srv = [("09:11:05", "web01: admin login from 198.51.100.7")]
    def sec(t): h, m, s = map(int, t.split(":")); return h * 3600 + m * 60 + s
    for t, msg in fw + srv:
        print(f"  raw {t}  {msg}")
    corrected = sec(srv[0][0]) + 240
    print(f"  corrected server time = {corrected // 3600:02d}:{corrected % 3600 // 60:02d}:{corrected % 60:02d}"
          f" -> login happened 3 s AFTER the firewall allowed the connection")

Source             offset ms  delay ms
LAN server             249.5      11.0
Internet server        250.0      80.0
Asymmetric path        275.0      70.0

Stratum: 0 = atomic/GPS clock, 1 = server wired to it, 2 = syncs from stratum 1 ...

Correlating firewall and server logs (server clock 240 s slow):
  raw 09:15:02  firewall: allowed 198.51.100.7 -> web01:443
  raw 09:11:05  web01: admin login from 198.51.100.7
  corrected server time = 09:15:05 -> login happened 3 s AFTER the firewall allowed the connection
